In [4]:
# ============================================
# Hyperparameter Tuning - Imports
# ============================================

import numpy as np
import pandas as pd
import joblib
import time

from sklearn.model_selection import RandomizedSearchCV

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

print("Imports completed successfully.")

Imports completed successfully.


In [5]:
# ============================================
# Load Zero-Day Dataset
# ============================================

X_train = joblib.load("../data/X_train_zero_day.pkl")
y_train = joblib.load("../data/y_train_zero_day.pkl")

X_train_scaled = joblib.load("../data/X_train_zero_day_scaled.pkl")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_train_scaled:", X_train_scaled.shape)

X_train: (125973, 122)
y_train: (125973,)
X_train_scaled: (125973, 122)


In [3]:
# ============================================
# Decision Tree - Hyperparameter Tuning
# ============================================

dt = DecisionTreeClassifier(
    random_state=42
)

dt_param_dist = {
    "criterion": ["gini", "entropy", "log_loss"],
    "max_depth": [None, 10, 15, 20, 25, 30, 40],
    "min_samples_split": [2, 5, 10, 20, 50],
    "min_samples_leaf": [1, 2, 5, 10, 20],
    "max_features": [None, "sqrt", "log2"]
}

dt_search = RandomizedSearchCV(
    estimator=dt,
    param_distributions=dt_param_dist,
    n_iter=20,
    scoring="f1",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

start_time = time.time()

dt_search.fit(X_train, y_train)

dt_tuning_time = time.time() - start_time

print("\nTuning completed!")
print("Tuning time:", round(dt_tuning_time, 2), "seconds")

Fitting 3 folds for each of 20 candidates, totalling 60 fits

Tuning completed!
Tuning time: 36.47 seconds


In [4]:
# ============================================
# Best Decision Tree Parameters
# ============================================

print("Best Parameters:")
print(dt_search.best_params_)

print("\nBest Cross-Validation F1:")
print(round(dt_search.best_score_, 4))

Best Parameters:
{'min_samples_split': 20, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 40, 'criterion': 'entropy'}

Best Cross-Validation F1:
0.9979


In [5]:
# ============================================
# Load Seen Test and Zero-Day Test
# ============================================

X_test_seen = joblib.load("../data/X_test_seen.pkl")
y_test_seen = joblib.load("../data/y_test_seen.pkl")

X_test_zero_day = joblib.load("../data/X_test_zero_day.pkl")
y_test_zero_day = joblib.load("../data/y_test_zero_day.pkl")

print("Seen Test:", X_test_seen.shape)
print("Zero-Day Test:", X_test_zero_day.shape)

print("\nSeen Test labels:")
print(pd.Series(y_test_seen).value_counts())

print("\nZero-Day Test labels:")
print(pd.Series(y_test_zero_day).value_counts())

Seen Test: (18794, 122)
Zero-Day Test: (3750, 122)

Seen Test labels:
binary_label
0    9711
1    9083
Name: count, dtype: int64

Zero-Day Test labels:
binary_label
1    3750
Name: count, dtype: int64


In [6]:
# ============================================
# Evaluate Tuned Decision Tree
# ============================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

best_dt = dt_search.best_estimator_

# -------------------------
# Seen Test
# -------------------------

start_time = time.time()

y_pred_seen_tuned = best_dt.predict(X_test_seen)

seen_prediction_time = time.time() - start_time


# -------------------------
# Zero-Day Test
# -------------------------

start_time = time.time()

y_pred_zero_day_tuned = best_dt.predict(X_test_zero_day)

zero_day_prediction_time = time.time() - start_time


# -------------------------
# Metrics - Seen Test
# -------------------------

seen_accuracy = accuracy_score(y_test_seen, y_pred_seen_tuned)
seen_precision = precision_score(y_test_seen, y_pred_seen_tuned)
seen_recall = recall_score(y_test_seen, y_pred_seen_tuned)
seen_f1 = f1_score(y_test_seen, y_pred_seen_tuned)


# -------------------------
# Zero-Day Detection
# -------------------------

zero_day_detection = np.mean(y_pred_zero_day_tuned == 1)


# -------------------------
# FPR on Normal samples
# -------------------------

normal_mask = (y_test_seen == 0)

false_positives = np.sum(
    y_pred_seen_tuned[normal_mask] == 1
)

total_normal = np.sum(normal_mask)

fpr = false_positives / total_normal


# -------------------------
# Print Results
# -------------------------

print("========== Tuned Decision Tree ==========")

print("\nSeen Test:")
print(f"Accuracy  : {seen_accuracy:.4f}")
print(f"Precision : {seen_precision:.4f}")
print(f"Recall    : {seen_recall:.4f}")
print(f"F1        : {seen_f1:.4f}")

print("\nZero-Day Test:")
print(f"Detection Rate : {zero_day_detection:.4f}")

print("\nFalse Positive Rate:")
print(f"FPR : {fpr:.4f}")

print("\nPrediction Time:")
print(f"Seen Test     : {seen_prediction_time:.4f} sec")
print(f"Zero-Day Test : {zero_day_prediction_time:.4f} sec")

========== Tuned Decision Tree ==========

Seen Test:
Accuracy  : 0.8796
Precision : 0.9617
Recall    : 0.7821
F1        : 0.8627

Zero-Day Test:
Detection Rate : 0.3664

False Positive Rate:
FPR : 0.0291

Prediction Time:
Seen Test     : 0.0212 sec
Zero-Day Test : 0.0035 sec


In [7]:
# ============================================
# Confusion Matrix - Tuned Decision Tree
# ============================================

cm = confusion_matrix(
    y_test_seen,
    y_pred_seen_tuned
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[9428  283]
 [1979 7104]]


In [6]:
# ============================================
# Random Forest - Hyperparameter Tuning
# ============================================

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

rf_param_dist = {
    "n_estimators": [100, 150, 200, 250, 300],
    "max_depth": [None, 10, 15, 20, 25, 30, 40],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": ["sqrt", "log2", None]
}

rf_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=rf_param_dist,
    n_iter=20,
    scoring="f1",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

start_time = time.time()

rf_search.fit(X_train, y_train)

rf_tuning_time = time.time() - start_time

print("\nTuning completed!")
print("Tuning time:", round(rf_tuning_time, 2), "seconds")

Fitting 3 folds for each of 20 candidates, totalling 60 fits


KeyboardInterrupt: 

In [9]:
# ============================================
# Best Random Forest Parameters
# ============================================

print("Best Parameters:")
print(rf_search.best_params_)

print("\nBest Cross-Validation F1:")
print(round(rf_search.best_score_, 4))

Best Parameters:
{'n_estimators': 250, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 40}

Best Cross-Validation F1:
0.9988


In [ ]:

# ============================================
# Evaluate Tuned Random Forest
# ============================================

best_rf = rf_search.best_estimator_

# -------------------------
# Seen Test
# -------------------------

start_time = time.time()

y_pred_seen_rf_tuned = best_rf.predict(X_test_seen)

rf_seen_prediction_time = time.time() - start_time


# -------------------------
# Zero-Day Test
# -------------------------

start_time = time.time()

y_pred_zero_day_rf_tuned = best_rf.predict(X_test_zero_day)

rf_zero_day_prediction_time = time.time() - start_time


# -------------------------
# Seen Test Metrics
# -------------------------

rf_seen_accuracy = accuracy_score(
    y_test_seen,
    y_pred_seen_rf_tuned
)

rf_seen_precision = precision_score(
    y_test_seen,
    y_pred_seen_rf_tuned
)

rf_seen_recall = recall_score(
    y_test_seen,
    y_pred_seen_rf_tuned
)

rf_seen_f1 = f1_score(
    y_test_seen,
    y_pred_seen_rf_tuned
)


# -------------------------
# Zero-Day Detection Rate
# -------------------------

rf_zero_day_detection = np.mean(
    y_pred_zero_day_rf_tuned == 1
)


# -------------------------
# False Positive Rate
# -------------------------

normal_mask = (y_test_seen == 0)

rf_false_positives = np.sum(
    y_pred_seen_rf_tuned[normal_mask] == 1
)

rf_total_normal = np.sum(normal_mask)

rf_fpr = (
    rf_false_positives /
    rf_total_normal
)


# -------------------------
# Results
# -------------------------

print("========== Tuned Random Forest ==========")

print("\nSeen Test:")
print(f"Accuracy  : {rf_seen_accuracy:.4f}")
print(f"Precision : {rf_seen_precision:.4f}")
print(f"Recall    : {rf_seen_recall:.4f}")
print(f"F1        : {rf_seen_f1:.4f}")

print("\nZero-Day Test:")
print(f"Detection Rate : {rf_zero_day_detection:.4f}")

print("\nFalse Positive Rate:")
print(f"FPR : {rf_fpr:.4f}")

print("\nPrediction Time:")
print(f"Seen Test     : {rf_seen_prediction_time:.4f} sec")
print(f"Zero-Day Test : {rf_zero_day_prediction_time:.4f} sec")

========== Tuned Random Forest ==========

Seen Test:
Accuracy  : 0.8758
Precision : 0.9604
Recall    : 0.7750
F1        : 0.8578

Zero-Day Test:
Detection Rate : 0.4411

False Positive Rate:
FPR : 0.0299

Prediction Time:
Seen Test     : 0.2949 sec
Zero-Day Test : 0.1452 sec


In [11]:
# ============================================
# Confusion Matrix - Tuned Random Forest
# ============================================

rf_cm = confusion_matrix(
    y_test_seen,
    y_pred_seen_rf_tuned
)

print("Confusion Matrix:")
print(rf_cm)

Confusion Matrix:
[[9421  290]
 [2044 7039]]


In [12]:
# ============================================
# KNN - Lightweight Hyperparameter Tuning
# ============================================

knn = KNeighborsClassifier(
    n_jobs=-1
)

knn_param_dist = {
    "n_neighbors": [3, 5, 7, 9, 11, 15],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"]
}

knn_search = RandomizedSearchCV(
    estimator=knn,
    param_distributions=knn_param_dist,
    n_iter=8,
    scoring="f1",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

start_time = time.time()

knn_search.fit(
    X_train_scaled,
    y_train
)

knn_tuning_time = time.time() - start_time

print("\nTuning completed!")
print(
    "Tuning time:",
    round(knn_tuning_time, 2),
    "seconds"
)

Fitting 3 folds for each of 8 candidates, totalling 24 fits

Tuning completed!
Tuning time: 1958.09 seconds


In [13]:
# ============================================
# Best KNN Parameters
# ============================================

print("Best Parameters:")
print(knn_search.best_params_)

print("\nBest Cross-Validation F1:")
print(round(knn_search.best_score_, 4))

Best Parameters:
{'weights': 'distance', 'n_neighbors': 3, 'metric': 'manhattan'}

Best Cross-Validation F1:
0.9971


In [15]:
import joblib

X_test_seen_scaled = joblib.load("../data/X_test_seen_scaled.pkl")
X_test_zero_day_scaled = joblib.load("../data/X_test_zero_day_scaled.pkl")

print("X_test_seen_scaled:", X_test_seen_scaled.shape)
print("X_test_zero_day_scaled:", X_test_zero_day_scaled.shape)

X_test_seen_scaled: (18794, 122)
X_test_zero_day_scaled: (3750, 122)


In [16]:
# ============================================
# Evaluate Tuned KNN
# ============================================

best_knn = knn_search.best_estimator_

# -------------------------
# Seen Test
# -------------------------

start_time = time.time()

y_pred_seen_knn_tuned = best_knn.predict(
    X_test_seen_scaled
)

knn_seen_prediction_time = time.time() - start_time


# -------------------------
# Zero-Day Test
# -------------------------

start_time = time.time()

y_pred_zero_day_knn_tuned = best_knn.predict(
    X_test_zero_day_scaled
)

knn_zero_day_prediction_time = time.time() - start_time


# -------------------------
# Seen Test Metrics
# -------------------------

knn_seen_accuracy = accuracy_score(
    y_test_seen,
    y_pred_seen_knn_tuned
)

knn_seen_precision = precision_score(
    y_test_seen,
    y_pred_seen_knn_tuned
)

knn_seen_recall = recall_score(
    y_test_seen,
    y_pred_seen_knn_tuned
)

knn_seen_f1 = f1_score(
    y_test_seen,
    y_pred_seen_knn_tuned
)


# -------------------------
# Zero-Day Detection
# -------------------------

knn_zero_day_detection = np.mean(
    y_pred_zero_day_knn_tuned == 1
)


# -------------------------
# False Positive Rate
# -------------------------

normal_mask = (y_test_seen == 0)

knn_false_positives = np.sum(
    y_pred_seen_knn_tuned[normal_mask] == 1
)

knn_total_normal = np.sum(normal_mask)

knn_fpr = (
    knn_false_positives /
    knn_total_normal
)


# -------------------------
# Results
# -------------------------

print("========== Tuned KNN ==========")

print("\nSeen Test:")
print(f"Accuracy  : {knn_seen_accuracy:.4f}")
print(f"Precision : {knn_seen_precision:.4f}")
print(f"Recall    : {knn_seen_recall:.4f}")
print(f"F1        : {knn_seen_f1:.4f}")

print("\nZero-Day Test:")
print(f"Detection Rate : {knn_zero_day_detection:.4f}")

print("\nFalse Positive Rate:")
print(f"FPR : {knn_fpr:.4f}")

print("\nPrediction Time:")
print(
    f"Seen Test     : {knn_seen_prediction_time:.4f} sec"
)
print(
    f"Zero-Day Test : {knn_zero_day_prediction_time:.4f} sec"
)

========== Tuned KNN ==========

Seen Test:
Accuracy  : 0.8552
Precision : 0.9495
Recall    : 0.7397
F1        : 0.8316

Zero-Day Test:
Detection Rate : 0.4077

False Positive Rate:
FPR : 0.0368

Prediction Time:
Seen Test     : 101.6112 sec
Zero-Day Test : 15.7623 sec


In [17]:
# ============================================
# Confusion Matrix - Tuned KNN
# ============================================

knn_cm = confusion_matrix(
    y_test_seen,
    y_pred_seen_knn_tuned
)

print("Confusion Matrix:")
print(knn_cm)

Confusion Matrix:
[[9354  357]
 [2364 6719]]


In [6]:
import joblib

X_train_zero_day_scaled = joblib.load("../data/X_train_zero_day_scaled.pkl")
y_train_zero_day = joblib.load("../data/y_train_zero_day.pkl")


In [7]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import LinearSVC
import time

svm = LinearSVC(
    random_state=42,
    max_iter=5000
)

svm_param_dist = {
    "C": [0.01, 0.1, 0.5, 1, 2, 5, 10],
    "class_weight": [None, "balanced"]
}

start_time = time.time()

svm_search = RandomizedSearchCV(
    estimator=svm,
    param_distributions=svm_param_dist,
    n_iter=6,
    scoring="f1",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

svm_search.fit(
    X_train_zero_day_scaled,
    y_train_zero_day
)

svm_tuning_time = time.time() - start_time

print("===== Tuned SVM =====")
print("Best Parameters:", svm_search.best_params_)
print(f"Best CV F1: {svm_search.best_score_:.4f}")
print(f"Tuning Time: {svm_tuning_time:.2f} sec")

Fitting 3 folds for each of 6 candidates, totalling 18 fits
===== Tuned SVM =====
Best Parameters: {'class_weight': 'balanced', 'C': 0.5}
Best CV F1: 0.9720
Tuning Time: 463.48 sec


In [ ]:
best_svm = svm_search.best_estimator_

# Seen Test
start_time = time.time()

y_pred_seen_svm_tuned = best_svm.predict(
    X_test_seen_scaled
)

svm_seen_prediction_time = time.time() - start_time


# Zero-Day Test
start_time = time.time()

y_pred_zero_day_svm_tuned = best_svm.predict(
    X_test_zero_day_scaled
)

svm_zero_day_prediction_time = time.time() - start_time


# Metrics
svm_seen_accuracy = accuracy_score(
    y_test_seen,
    y_pred_seen_svm_tuned
)

svm_seen_precision = precision_score(
    y_test_seen,
    y_pred_seen_svm_tuned,
    pos_label=1
)

svm_seen_recall = recall_score(
    y_test_seen,
    y_pred_seen_svm_tuned,
    pos_label=1
)

svm_seen_f1 = f1_score(
    y_test_seen,
    y_pred_seen_svm_tuned,
    pos_label=1
)

svm_zero_day_detection = np.mean(
    y_pred_zero_day_svm_tuned == 1
)


# FPR
false_positives = np.sum(
    (y_test_seen == 0) &
    (y_pred_seen_svm_tuned == 1)
)

total_normal = np.sum(
    y_test_seen == 0
)

svm_fpr = false_positives / total_normal


print("========== Tuned SVM ==========")

print(f"Seen Test:")
print(f"Accuracy  : {svm_seen_accuracy:.4f}")
print(f"Precision : {svm_seen_precision:.4f}")
print(f"Recall    : {svm_seen_recall:.4f}")
print(f"F1        : {svm_seen_f1:.4f}")

print(f"\nZero-Day Test:")
print(f"Detection Rate : {svm_zero_day_detection:.4f}")

print(f"\nFalse Positive Rate:")
print(f"FPR : {svm_fpr:.4f}")

print(f"\nPrediction Time:")
print(f"Seen Test     : {svm_seen_prediction_time:.4f} sec")
print(f"Zero-Day Test : {svm_zero_day_prediction_time:.4f} sec")

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_seen,
        y_pred_seen_svm_tuned
    )
)

========== Tuned SVM ==========
Seen Test:
Accuracy  : 0.8219
Precision : 0.8981
Recall    : 0.7122
F1        : 0.7944

Zero-Day Test:
Detection Rate : 0.3701

False Positive Rate:
FPR : 0.0756

Prediction Time:
Seen Test     : 0.0692 sec
Zero-Day Test : 0.0142 sec

Confusion Matrix:
[[8977  734]
 [2614 6469]]


In [24]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logreg_param_dist = {
    "C": [0.01, 0.1, 0.5, 1, 2, 5, 10],
    "class_weight": [None, "balanced"],
    "solver": ["lbfgs", "liblinear"]
}

start_time = time.time()

logreg_search = RandomizedSearchCV(
    estimator=logreg,
    param_distributions=logreg_param_dist,
    n_iter=6,
    scoring="f1",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

logreg_search.fit(
    X_train_zero_day_scaled,
    y_train_zero_day
)

logreg_tuning_time = time.time() - start_time

print("===== Tuned Logistic Regression =====")
print("Best Parameters:", logreg_search.best_params_)
print(f"Best CV F1: {logreg_search.best_score_:.4f}")
print(f"Tuning Time: {logreg_tuning_time:.2f} sec")

Fitting 3 folds for each of 6 candidates, totalling 18 fits
===== Tuned Logistic Regression =====
Best Parameters: {'solver': 'liblinear', 'class_weight': None, 'C': 10}
Best CV F1: 0.9725
Tuning Time: 379.71 sec


In [25]:
best_logreg = logreg_search.best_estimator_


# Seen Test
start_time = time.time()

y_pred_seen_logreg_tuned = best_logreg.predict(
    X_test_seen_scaled
)

logreg_seen_prediction_time = time.time() - start_time


# Zero-Day Test
start_time = time.time()

y_pred_zero_day_logreg_tuned = best_logreg.predict(
    X_test_zero_day_scaled
)

logreg_zero_day_prediction_time = time.time() - start_time


# Metrics
logreg_seen_accuracy = accuracy_score(
    y_test_seen,
    y_pred_seen_logreg_tuned
)

logreg_seen_precision = precision_score(
    y_test_seen,
    y_pred_seen_logreg_tuned,
    pos_label=1
)

logreg_seen_recall = recall_score(
    y_test_seen,
    y_pred_seen_logreg_tuned,
    pos_label=1
)

logreg_seen_f1 = f1_score(
    y_test_seen,
    y_pred_seen_logreg_tuned,
    pos_label=1
)

logreg_zero_day_detection = np.mean(
    y_pred_zero_day_logreg_tuned == 1
)


# FPR
false_positives = np.sum(
    (y_test_seen == 0) &
    (y_pred_seen_logreg_tuned == 1)
)

total_normal = np.sum(
    y_test_seen == 0
)

logreg_fpr = false_positives / total_normal


print("========== Tuned Logistic Regression ==========")

print(f"Seen Test:")
print(f"Accuracy  : {logreg_seen_accuracy:.4f}")
print(f"Precision : {logreg_seen_precision:.4f}")
print(f"Recall    : {logreg_seen_recall:.4f}")
print(f"F1        : {logreg_seen_f1:.4f}")

print(f"\nZero-Day Test:")
print(f"Detection Rate : {logreg_zero_day_detection:.4f}")

print(f"\nFalse Positive Rate:")
print(f"FPR : {logreg_fpr:.4f}")

print(f"\nPrediction Time:")
print(f"Seen Test     : {logreg_seen_prediction_time:.4f} sec")
print(f"Zero-Day Test : {logreg_zero_day_prediction_time:.4f} sec")

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_seen,
        y_pred_seen_logreg_tuned
    )
)

========== Tuned Logistic Regression ==========
Seen Test:
Accuracy  : 0.8233
Precision : 0.9002
Recall    : 0.7136
F1        : 0.7961

Zero-Day Test:
Detection Rate : 0.4091

False Positive Rate:
FPR : 0.0740

Prediction Time:
Seen Test     : 0.0371 sec
Zero-Day Test : 0.0061 sec

Confusion Matrix:
[[8992  719]
 [2601 6482]]
